# Trenujemy mini-GPT na recenzjach win

## Zadanie
Zaimplementuj od zera mały model autoregresyjny w PyTorchu. Model przewiduje następny token, uczy się na recenzjach win i generuje nowe opisy. Przećwiczymy pary wejście–cel, cross-entropy, pętlę treningową, perplexity i sampling. Mały zbiór demonstruje mechanikę, nie jakość produkcyjną.

## 1. Dane i tokenizacja
Korpus jest offline i dydaktyczny; podmień listę na lokalną kolumnę opisów Wine Reviews CSV. Zachowaj osobne train/validation/test i filtruj braki. Tokenizacja wyrazowa jest uproszczeniem; modele praktyczne używają subword (np. BPE).

In [ ]:
import math,re,random,torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(17);random.seed(17);device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
reviews=["aromatic cherry and plum with a long finish","bright acidity balances ripe citrus fruit","dry wine with notes of pear and mineral","soft tannins frame dark berry and spice","fresh apple aroma with crisp acidity","rich oak vanilla and blackcurrant flavors","light bodied wine with floral notes","ripe peach and honey with a smooth finish","structured tannins and earthy cherry notes","zesty lemon aroma and clean mineral finish","bold dark fruit with pepper and toasted oak","delicate strawberry notes and lively acidity","crisp white wine shows lime and green apple","deep ruby color with plum and cocoa","silky texture and a lingering herbal finish","balanced acidity with red cherry aroma","lemon zest meets saline mineral character","blackberry jam with firm structured tannins","fresh floral bouquet and subtle pear flavor","warm spice and vanilla soften ripe plum","elegant wine with bright fruit and fine acidity","earthy notes complement concentrated dark fruit","peach blossom and citrus lead to a clean finish","medium bodied red with cherry and cedar"]
def tok(s):return re.findall(r"[a-z]+|[^\w\s]",s.lower())
special=["<pad>","<unk>","<bos>","<eos>"];vocab=special+sorted({t for s in reviews for t in tok(s)});stoi={t:i for i,t in enumerate(vocab)};PAD=0
seq=[torch.tensor([2]+[stoi.get(t,1) for t in tok(s)]+[3]) for s in reviews]
random.shuffle(seq);cut=int(.8*len(seq));train,val=seq[:cut],seq[cut:]
def collate(rows):
    n=max(map(len,rows));x=torch.full((len(rows),n-1),PAD,dtype=torch.long);y=x.clone()
    for i,s in enumerate(rows):x[i,:len(s)-1]=s[:-1];y[i,:len(s)-1]=s[1:]
    return x.to(device),y.to(device)
x,y=collate(train[:4]);print("train",len(train),"val",len(val),"vocab",len(vocab),"x/y",x.shape,y.shape)

## 2. Cel next-token prediction
Sekwencję $[x_0,…,x_T]$ dzielimy na input $[x_0,…,x_{T-1}]$ i target $[x_1,…,x_T]$. Minimalizujemy $-Σ_t log p(x_{t+1}|x_{≤t})$. Padding ignorujemy przez `ignore_index`.

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self,d,h,drop):
        super().__init__();assert d%h==0;self.h=h;self.dh=d//h;self.qkv=nn.Linear(d,3*d);self.proj=nn.Linear(d,d);self.drop=nn.Dropout(drop)
    def forward(self,x):
        B,T,C=x.shape;q,k,v=self.qkv(x).chunk(3,-1)
        def s(z):return z.view(B,T,self.h,self.dh).transpose(1,2)
        q,k,v=map(s,(q,k,v));score=q@k.transpose(-2,-1)/math.sqrt(self.dh)
        mask=torch.tril(torch.ones(T,T,dtype=torch.bool,device=x.device));score=score.masked_fill(~mask,torch.finfo(score.dtype).min)
        return self.drop(self.proj((score.softmax(-1)@v).transpose(1,2).contiguous().view(B,T,C)))
class Block(nn.Module):
    def __init__(self,d,h,drop):
        super().__init__();self.n1=nn.LayerNorm(d);self.a=CausalAttention(d,h,drop);self.n2=nn.LayerNorm(d);self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d),nn.Dropout(drop))
    def forward(self,x):x=x+self.a(self.n1(x));return x+self.ff(self.n2(x))
class MiniGPT(nn.Module):
    def __init__(self,V,maxlen,d=64,h=4,layers=2,drop=.1):
        super().__init__();self.maxlen=maxlen;self.tok=nn.Embedding(V,d);self.pos=nn.Embedding(maxlen,d);self.blocks=nn.ModuleList([Block(d,h,drop) for _ in range(layers)]);self.ln=nn.LayerNorm(d);self.head=nn.Linear(d,V)
    def forward(self,idx):
        idx=idx[:,-self.maxlen:];T=idx.size(1);p=torch.arange(T,device=idx.device);z=self.tok(idx)+self.pos(p)[None]
        for b in self.blocks:z=b(z)
        return self.head(self.ln(z))
maxlen=max(len(s) for s in seq)-1;model=MiniGPT(len(vocab),maxlen).to(device)
print("parametry",sum(p.numel() for p in model.parameters()),"logity",model(x).shape)

## 3. Trening, cross-entropy i perplexity
Perplexity $exp(L)$ jest wykładniczą średnią ujemnego log-prawdopodobieństwa tokenu. Niższa jest lepsza przy tym samym tokenizerze i zbiorze; różnych tokenizerów nie porównuj bezpośrednio.

In [ ]:
@torch.no_grad()
def evaluate(data):
    model.eval();ls=[]
    for i in range(0,len(data),8):
        a,b=collate(data[i:i+8]);ls.append(F.cross_entropy(model(a).reshape(-1,len(vocab)),b.reshape(-1),ignore_index=PAD).item())
    model.train();return sum(ls)/len(ls)
opt=torch.optim.AdamW(model.parameters(),lr=3e-3,weight_decay=.01);tr=[];va=[]
for step in range(201):
    a,b=collate(random.sample(train,min(8,len(train))));logits=model(a)
    loss=F.cross_entropy(logits.reshape(-1,len(vocab)),b.reshape(-1),ignore_index=PAD)
    opt.zero_grad(set_to_none=True);loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()
    if step%20==0:
        vl=evaluate(val);tr.append(loss.item());va.append(vl);print(step,f"train={loss.item():.3f} val={vl:.3f} PPL={math.exp(vl):.2f}")
import matplotlib.pyplot as plt
plt.plot(range(0,201,20),tr,label="train batch");plt.plot(range(0,201,20),va,label="validation");plt.xlabel("krok");plt.ylabel("loss");plt.legend();plt.grid();plt.show()

## 4. Generowanie
Generujemy iteracyjnie z $p(x_t|x_{<t})$. Temperatura $τ$ dzieli logity przez $τ$: niska koncentruje rozkład, wysoka go spłaszcza. Top-k usuwa tokeny poza k najlepszymi.

In [ ]:
@torch.no_grad()
def generate(prompt="bright acidity",steps=18,temperature=.8,top_k=8):
    model.eval();ids=[2]+[stoi.get(t,1) for t in tok(prompt)];idx=torch.tensor(ids,device=device)[None]
    for _ in range(steps):
        logits=model(idx)[:,-1,:]/max(temperature,1e-5);logits[:,PAD]=-float("inf")
        if top_k:
            threshold=torch.topk(logits,min(top_k,logits.size(-1))).values[:,-1,None];logits=logits.masked_fill(logits<threshold,-float("inf"))
        nxt=torch.multinomial(logits.softmax(-1),1);idx=torch.cat([idx,nxt],1)
        if nxt.item()==3:break
    return " ".join(vocab[i] for i in idx[0].tolist() if vocab[i] not in special)
for t in (.4,.8,1.2):print(f"T={t}:",generate(temperature=t))

## 5. Analiza i rozszerzenia
Mały korpus sprzyja zapamiętywaniu. Walidacja pozwala zauważyć rozjazd train/validation, ale wynik jest niestabilny.

1. Zapisuj checkpoint o najlepszej stracie walidacyjnej i dodaj early stopping.
2. Zaimplementuj nucleus sampling top-p.
3. Podmień tokenizer na subword i porównaj długości oraz PPL.
4. Wczytaj lokalne Wine Reviews CSV, filtruj braki i zachowaj test set.
5. Zapisz generacje, ustawienia dekodowania i przykłady błędów.

## Podsumowanie
Teacher forcing liczy stratę na wszystkich prefiksach równolegle, a maska causal zapobiega przeciekowi. Cross-entropy jest celem, PPL jego transformacją. Sampling steruje różnorodnością, lecz nie naprawia danych ani ograniczeń modelu.